<a href="https://colab.research.google.com/github/rpp5069-cpu/job-pipeline/blob/main/CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS / MBA / Data Analytics / AI
## USAJobs + Jooble — Production Collection & O\\*NET 24.2 Competency Mapper
### Google Colab + Google Drive | Three Automation Backends
**Penn State Great Valley | D.Eng. Research — AI in the Workforce & Labor Market**
--## Pipeline flow
```
USAJobs API ──┐
├──► Section 4–5: Live Deep Fetch
Jooble API ──┘
│
Section 6: Score + Merge + Dedup
│
Section 3B: O*NET 24.2 (35 skills)
│
Section 7: Save to Google Drive
Research/JobPostings/
├── outputs/ MASTER_YYYYMMDD_HHMM.{json,csv,xlsx}
└── archive/ (prior runs)
```
**Usage:** `Runtime → Run all`. Drive auth dialog appears once; everything runs
automatically.
| Automation option | Where configured |
|---|---|

| **A — Google Apps Script** (simplest) | Section 8-A markdown — copy-paste JS into
script.google.com |
| **B — GitHub Actions** (most reliable free) | Section 8-B markdown — copy-paste YAML
into repo |
| **C — GCP Cloud Scheduler + Cloud Run** (production) | Section 8-C markdown — gcloud
commands |
**API keys required** (add once in Colab Secrets

🔑)

| Secret | Source |
|---|---|
| `USAJOBS_EMAIL` | Registered email at developer.usajobs.gov |
| `USAJOBS_KEY` | API key from the same registration |
| `JOOBLE_KEY` | Free key at jooble.org/api (500 req/day) |

In [ ]:
!pip install -q pandas openpyxl requests beautifulsoup4 tqdm
print("Packages ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 9.9 MB/s eta 0:00:00
Packages ready.


In [ ]:
import re, json, os, hashlib, time, shutil, warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from collections import Counter
from itertools import chain
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
# ── Credential loader: works in Colab AND in Papermill / GitHub Actions ───────
def _get_secret(key):
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        return os.environ.get(key, '')
USAJOBS_EMAIL = _get_secret('USAJOBS_EMAIL')
USAJOBS_KEY = _get_secret('USAJOBS_KEY')
JOOBLE_KEY = '4ad82ea7-0118-4f3a-8825-20ed48373e17'
CURRENT_EPOCH = datetime(2026, 1, 1)
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M')
ONET_VERSION = 'O*NET 24.2'
MASTER_JSON = f'MASTER_{RUN_TS}.json'
MASTER_CSV = f'MASTER_{RUN_TS}.csv'
MASTER_XLSX = f'MASTER_{RUN_TS}.xlsx'


print(f"Imports complete. Run: {RUN_TS} | Epoch: {CURRENT_EPOCH.date()}")
print(f" USAJOBS_EMAIL : {'✓' if USAJOBS_EMAIL else '✗ MISSING'}")
print(f" USAJOBS_KEY : {'✓' if USAJOBS_KEY else '✗ MISSING'}")
print(f" JOOBLE_KEY : {'✓' if JOOBLE_KEY else '✗ MISSING'}")

Imports complete. Run: 20260624_1420 | Epoch: 2026-01-01
 USAJOBS_EMAIL : ✓
 USAJOBS_KEY : ✓
 JOOBLE_KEY : ✓


--## Section 1 — Configuration
Expand or trim keyword lists to adjust coverage. Default settings target maximum
volume within free-tier quotas:
| Source | Keywords | Pages | Results/page | Max raw |
|--------|----------|-------|-------------|---------|
| USAJobs | 37 | 10 | 25 | ~9,250 |
| Jooble | 33 | 5 | ~20 | ~3,300 |

In [ ]:
# --- USAJobs keyword map --- 4 domains
USAJOBS_QUERIES = {
'CS': [
'software engineer', 'computer scientist',
'information technology specialist', 'cybersecurity analyst',
'cloud engineer', 'devops engineer', 'systems administrator',
'network engineer', 'platform engineer', 'site reliability engineer',
'solutions architect', 'security engineer',
],
'MBA': [
'management analyst', 'program analyst',
'operations research analyst', 'financial analyst',
'budget analyst', 'management and program analyst',
'administrative officer', 'product manager',
'strategy analyst', 'corporate strategy',
],
'DATA_ANALYTICS': [
'data analyst', 'statistician', 'data scientist',
'business intelligence analyst', 'quantitative analyst',
'data engineer', 'reporting analyst', 'analytics engineer',
'senior data analyst', 'data visualization analyst',
],
'AI': [
'artificial intelligence', 'machine learning engineer',
'natural language processing', 'computer vision engineer',
'applied scientist', 'MLOps engineer', 'deep learning engineer',
'AI researcher', 'generative AI engineer', 'NLP engineer',
'LLM engineer', 'AI product manager', 'reinforcement learning',
'research scientist', 'data scientist machine learning',
],
}


# --- Jooble keyword map --- 4 domains
JOOBLE_QUERIES = {
'CS': [
'software engineer', 'DevOps engineer', 'cloud engineer',
'backend developer', 'full stack developer', 'cybersecurity analyst',
'systems architect', 'platform engineer', 'solutions architect',
'security engineer', 'network engineer',
],
'MBA': [
'strategy consultant', 'product manager', 'business analyst',
'operations manager', 'management consultant', 'financial analyst',
'corporate strategy', 'program manager', 'management analyst',
],
'DATA_ANALYTICS': [
'data analyst', 'business intelligence analyst', 'analytics engineer',
'data engineer', 'BI developer', 'quantitative analyst',
'data visualization analyst', 'reporting analyst', 'statistician',
],
'AI': [
'machine learning engineer', 'AI researcher', 'NLP engineer',
'MLOps engineer', 'generative AI engineer', 'applied scientist',
'deep learning engineer', 'LLM engineer', 'computer vision engineer',
'reinforcement learning engineer', 'AI product manager',
'prompt engineer', 'data scientist',
],
}
# --- Pagination ---
USAJOBS_PAGE_SIZE = 25
# max 500; 25 is well within rate limits
USAJOBS_MAX_PAGES = 10
# 10 × 25 = 250 per keyword
JOOBLE_MAX_PAGES = 5
# ~20 results per page × 5 = 100 per keyword
# --- Rate limiting (seconds between consecutive API calls) ---
USAJOBS_DELAY = 1.0
JOOBLE_DELAY = 0.8
# --- Minimum relevance score to keep a record ---
MIN_SCORE = 2
usa_kw = sum(len(v) for v in USAJOBS_QUERIES.values())
jbl_kw = sum(len(v) for v in JOOBLE_QUERIES.values())
print(f"Configuration ready.")
print(f" USAJobs : {usa_kw} keywords × {USAJOBS_MAX_PAGES} pages ×\n{USAJOBS_PAGE_SIZE} = up to {usa_kw*USAJOBS_MAX_PAGES*USAJOBS_PAGE_SIZE:,} raw")
print(f" Jooble\n: {jbl_kw} keywords × {JOOBLE_MAX_PAGES} pages × ~20\n= up to\n{jbl_kw*JOOBLE_MAX_PAGES*20:,} raw")

Configuration ready.
 USAJobs : 47 keywords × 10 pages ×
25 = up to 11,750 raw
 Jooble
: 42 keywords × 5 pages × ~20
= up to
4,200 raw


---

## Section 2 — Google Drive Mount & Folder Setup
Drive is mounted at `/content/drive`. All output files are copied there directly
via `shutil.copy2` — no extra API credentials needed.
Change `DRIVE_BASE` if your folder is located elsewhere in My Drive.

In [20]:
import os, shutil
import pandas as pd

# --- Environment-Aware Drive Setup ---
try:
    from google.colab import drive as colab_drive
    colab_drive.mount('/content/drive', force_remount=False)
    print("Google Drive mounted at /content/drive")
    IS_COLAB = True
except ImportError:
    print("Running outside Colab (Headless/GitHub Actions) - Skipping Drive Mount")
    IS_COLAB = False

# --- Papermill Parameters Cell ---
# The 'parameters' tag MUST be added to this cell's metadata.
# Default path for Colab; GitHub Actions will override this via the workflow YAML.
DRIVE_BASE = globals().get('DRIVE_BASE', '/content/drive/MyDrive/Research/JobPostings')

DRIVE_OUTPUTS = os.path.join(DRIVE_BASE, 'outputs')
DRIVE_ARCHIVE = os.path.join(DRIVE_BASE, 'archive')
DRIVE_LOGS = os.path.join(DRIVE_BASE, 'logs')
LOCAL_WORK = './job_pipeline_work'

for folder in [DRIVE_OUTPUTS, DRIVE_ARCHIVE, DRIVE_LOGS, LOCAL_WORK]:
    os.makedirs(folder, exist_ok=True)

print(f"outputs : {DRIVE_OUTPUTS}")
print(f"archive : {DRIVE_ARCHIVE}")

def save_to_drive(local_path: str, drive_dir: str, fname: str) -> str:
    """Copy a local file to the destination directory."""
    dst = os.path.join(drive_dir, fname)
    shutil.copy2(local_path, dst)
    kb = os.path.getsize(dst) / 1024
    print(f"{fname} ({kb:.1f} KB) → {drive_dir} ✅")
    return dst

def write_run_log(df: pd.DataFrame) -> None:
    log = os.path.join(DRIVE_LOGS, 'run_log.tsv')
    needs_header = not os.path.exists(log)
    RUN_TS = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
    with open(log, 'a', encoding='utf-8') as fh:
        if needs_header:
            fh.write('run_ts\ttotal\tcurrent\thistoric\tUSAJobs\tJooble\thigh\tmedium\tlow\n')
        fh.write('\t'.join([
            RUN_TS, str(len(df)),
            str((df.era=='current').sum()), str((df.era=='historical').sum()),
            str((df.source=='USAJobs').sum()), str((df.source=='Jooble').sum()),
            str((df.relevance_tier=='HIGH').sum()),
            str((df.relevance_tier=='MEDIUM').sum()),
            str((df.relevance_tier=='LOW').sum()),
        ]) + '\n')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive
outputs : /content/drive/MyDrive/Research/JobPostings/outputs
archive : /content/drive/MyDrive/Research/JobPostings/archive


In [22]:
import json
import os

def add_parameters_tag_to_s2():
    """Programmatically adds the 'parameters' tag to the s2-drive cell."""
    print("Applying 'parameters' tag to cell 's2-drive'...")

    try:
        from google.colab import _message
        # We use a Colab internal command to update the metadata of cell 's2-drive'
        _message.blocking_request(
            'set_cell_metadata',
            request={'cellId': 's2-drive', 'metadata': {'tags': ['parameters']}},
            timeout_sec=5
        )
        print("✅ Success! The 'parameters' tag has been added to the 's2-drive' cell.")
        print("ACTION REQUIRED: Press Ctrl+S (or Cmd+S) NOW to save this change to the file.")
    except ImportError:
        print("⚠️ Running outside Colab environment - skipping tag application.")
    except Exception as e:
        print(f"❌ Could not apply tag automatically: {e}")
        print("Please try the 'Command Palette' (Ctrl+Shift+P) and search for 'Edit cell metadata' one last time.")

add_parameters_tag_to_s2()

Applying 'parameters' tag to cell 's2-drive'...
✅ Success! The 'parameters' tag has been added to the 's2-drive' cell.
ACTION REQUIRED: Press Ctrl+S (or Cmd+S) NOW to save this change to the file.


--## Section 3A — Domain Relevance Scoring Engine & Utilities
| Function | Purpose |
|---|---|
| `score_relevance()` | 0–12 score; HIGH ≥ 6 / MEDIUM ≥ 4 / LOW ≥ 2 / EXCLUDE < 2 |
| `extract_skills()` | Technical tools & languages (SKILL_TAXONOMY) |
| `infer_domain()` | CS / MBA / DATA_ANALYTICS / AI |
| `classify_era()` | `current` (≥ Jan 2026) or `historical` |
| `make_hash()` | SHA-1 of title+company+location for cross-source dedup |
| `normalize_location()` | Standardize to `City, ST, US` format |

In [ ]:
# --- Technical skill taxonomy
SKILL_TAXONOMY = [
'Python','R','SQL','Java','Scala','Go','JavaScript','TypeScript','C++','Rust','MATLAB',
'Julia',
'PyTorch','TensorFlow','Keras','Scikit-learn','XGBoost','LightGBM','Hugging Face','JAX','ONNX','LangChain',
'Machine Learning','Deep Learning','NLP','Computer Vision','Reinforcement Learning','MLOps',
'Feature Engineering','Model Deployment','A/B Testing','Large Language Models',
'Generative AI','RAG','Prompt Engineering','Fine-tuning','RLHF',
'Spark','Kafka','Airflow','dbt','Databricks','Snowflake','BigQuery','Redshift',
'ETL','ELT','Data Warehouse','Data Lake','Delta Lake','Flink',
'Tableau','Power BI','Looker','Qlik','Excel','Statistics','Pandas','NumPy',
'Matplotlib','Seaborn','Plotly',
'AWS','Azure','GCP','Docker','Kubernetes','Terraform','Helm','ArgoCD',
'REST API','GraphQL','CI/CD','Git','Linux','Microservices','Cybersecurity','gRPC',
'Financial Modeling','Six Sigma','Agile','Scrum','Project Management',
'Strategic Planning','Business Intelligence','OKRs','Stakeholder Management',
]


DOMAIN_TITLE_T1 = [
'data analyst','senior data analyst','analytics engineer','business intelligence',
'bi developer','bi analyst','quantitative analyst','data visualization','reporting analyst',
'data engineer','machine learning engineer','ml engineer','applied scientist',
'research scientist','ai engineer','nlp engineer','deep learning engineer',
'computer vision engineer','mlops engineer','data scientist','ai researcher',
'generative ai','llm engineer','natural language processing','reinforcement learning',
'software engineer','software developer','systems architect','cloud engineer',
'devops engineer','platform engineer','site reliability engineer','backend engineer',
'cybersecurity analyst','security engineer','solutions architect','computer scientist',
'network engineer','systems administrator','information technology specialist',
'strategy consultant','management consultant','business analyst','product manager',
'operations manager','financial analyst','management analyst','operations analyst',
'program analyst','budget analyst','management and program analyst','program manager',
'prompt engineer','llm engineer','ai product manager',
]
DOMAIN_TITLE_T2 = [
'engineer','developer','analyst','scientist','architect','consultant','manager',
'director','specialist','python','data','cloud','ai','ml','machine learning',
'analytics','intelligence','research','technology','software','information technology',
]
DOMAIN_DESC_TERMS = [
'machine learning','deep learning','neural network','nlp','natural language processing',
'computer vision','pytorch','tensorflow','scikit-learn','hugging face','llm',
'large language model','generative ai','rag','mlops','python','sql','scala','java',
'data pipeline','etl','data warehouse','spark','kafka','airflow','dbt',
'databricks','snowflake','bigquery','redshift','tableau','power bi','looker',
'aws','azure','gcp','docker','kubernetes','terraform','a/b testing','statistics',
'mba','strategy','financial modeling','agile','scrum','rest api','devops',
'cybersecurity','information security','federal','clearance',
]
DOMAIN_ORG_NAMES = [
'department of defense','dod','national security agency','nsa','nasa',
'national science foundation','department of energy','department of veterans affairs',
'centers for disease control','cdc','department of treasury','irs',
'general services administration','gsa','department of commerce','census bureau',
'department of homeland security','department of state','department of labor',
'google','amazon','meta','apple','microsoft','openai','anthropic','deepmind',
'nvidia','intel','ibm','oracle','salesforce','databricks','snowflake','palantir',
'mckinsey','bcg','bain','deloitte','accenture','pwc','booz allen','leidos','mitre',
'jpmorgan','goldman sachs','bloomberg',
]
def _pat(terms):
    esc = [re.escape(t) for t in sorted(terms, key=len, reverse=True)]
    return re.compile(r'\b(' + '|'.join(esc) + r')\b', re.IGNORECASE)
PAT_T1 = _pat(DOMAIN_TITLE_T1)
PAT_T2 = _pat(DOMAIN_TITLE_T2)
PAT_DD = _pat(DOMAIN_DESC_TERMS)
PAT_OR = _pat(DOMAIN_ORG_NAMES)

def score_relevance(title='', description='', company=''):
    score, matched = 0, []
    tl = str(title).lower()
    dl = str(description).lower()[:3000]
    cl = str(company).lower()
    for pat, pts in [(PAT_T1, 4), (PAT_T2, 2)]:
        h = pat.findall(tl)
        if h: score += min(pts, pts * len(set(h))); matched += list(set(h))
    h = PAT_DD.findall(dl)
    if h: score += min(4, len(set(h))); matched += list(set(h))
    h = PAT_OR.findall(cl)
    if h: score += 2; matched += list(set(h))
    tier = ('HIGH' if score >= 6 else
            'MEDIUM' if score >= 4 else
            'LOW' if score >= 2 else
            'EXCLUDE')
    return score, list(set(matched)), tier

def extract_skills(title='', description=''):
    c = (str(title) + ' ' + str(description)).lower()
    return [s for s in SKILL_TAXONOMY if s.lower() in c]

def infer_domain(title='', description=''):
    t = (str(title) + ' ' + str(description)).lower()
    if any(k in t for k in ['machine learning','deep learning','nlp','neural','pytorch',
    'tensorflow','mlops','applied scientist','llm','generative','computer vision',
    'reinforcement','artificial intelligence']): return 'AI'
    if any(k in t for k in ['data analyst','business intelligence','analytics engineer',
    'tableau','power bi','looker','quantitative analyst','statistician']): return 'DATA_ANALYTICS'
    if any(k in t for k in ['mba','strategy','consultant','operations manager',
    'product manager','financial analyst','management analyst','budget analyst',
    'program analyst','program manager']): return 'MBA'
    return 'CS'

def classify_era(date_str):
    if not date_str: return 'unknown'
    try:
        return ('current' if datetime.fromisoformat(str(date_str)[:10]) >= CURRENT_EPOCH else 'historical')
    except Exception: return 'unknown'

def make_hash(title, company, location):
    key = \
    f'{str(title).lower().strip()}|{str(company).lower().strip()}|{str(location).lower().strip()}'
    return hashlib.sha1(key.encode()).hexdigest()[:12]

def clean_html(text):
    if not text or isinstance(text, float): return ''
    return BeautifulSoup(str(text), 'html.parser').get_text(separator=' ', strip=True)

def normalize_location(raw):
    if not raw: return 'Unknown, US'
    raw = str(raw).strip()
    if ', US' in raw: return raw
    STATE_MAP = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA',
    'Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD',
    'Massachusetts':'MA','MA':'MA','Minnesota':'MN','Mississippi':'MS',
    'Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY',
    'North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK',
    'Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC',
    'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT',
    'Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV',
    'Wisconsin':'WI','Wyoming':'WY','District of Columbia':'DC',
    }
    for full, abbr in STATE_MAP.items():
        if full in raw:
            parts = raw.split(',')
            city = parts[0].strip() if len(parts) > 1 else ''
            return f'{city}, {abbr}, US' if city else f'{abbr}, US'
    if 'remote' in raw.lower(): return 'Remote, US'
    return f'{raw}, US'


print("Scoring engine ready.")

Scoring engine ready.


--## Section 3B — O\\*NET 24.2 Competency Framework (35 skills)
Every posting receives two skill fields:
| Field | Contents |
|---|---|
| `required_skills` | Technical tools extracted from job text (Python, SQL, …) |
| `onet_skills` | Up to 35 O\\*NET 24.2 standardised competency names |
Mapping uses keyword/phrase triggers from the official O\\*NET Content Model.
`map_onet_skills()` is called automatically inside both API parsers.

In [ ]:
ONET_SKILLS_MAP = {
# ── BASIC SKILLS ──────────────────────────────────────────────────────────
"Reading Comprehension": ["documentation","technical document","specification",
"interpret report","review document","written material","policy document",
"literature review","parse","technical manual"],
"Active Listening": ["active listening","stakeholder","requirements gathering",
"client meetings","user interview","discovery session","elicit requirements"],
"Writing": ["writing","written communication","technical writing","report",
"proposal","white paper","draft","document findings","publish"],
"Speaking": ["presentation","verbal communication","briefing","communicate ideas",
"public speaking","articulate","cross-functional communication"],
"Mathematics":
["math","mathematics","statistics","quantitative","calculus","algebra",
"numerical analysis","probability","linear algebra","regression","optimization"],
"Science": ["scientific","research","experiment","hypothesis","empirical",
"research methodology","scientific method","data collection","study design"],
"Critical Thinking": ["critical thinking","analytical","analysis","evaluate","assess",
"problem-solving","logical","reasoning","judgment","root cause","synthesize",
"evidence-based"],
"Active Learning": ["active learning","continuous learning","self-directed",
"upskill","reskill","adapt","emerging technology","stay current",
"professional development","evolving landscape"],
"Learning Strategies": ["learning strategies","training program","curriculum",
"instructional design","e-learning","onboarding plan","coaching","workshop"],
"Monitoring": ["monitor","track","KPI","metrics","performance monitoring",
"dashboard","alerting","observability","logging","SLA","telemetry"],
# ── SOCIAL SKILLS ─────────────────────────────────────────────────────────
"Social Perceptiveness": ["social perceptiveness","emotional intelligence","empathy",
"interpersonal","team dynamics","cultural awareness","relationship building"],
"Coordination": ["coordination","coordinate","cross-functional","collaboration",
"teamwork","liaison","partner with","multi-team","synchronize"],
"Persuasion": ["persuasion","influence","sales","convince","advocate",
"change management","buy-in","executive presence","drive adoption"],
"Negotiation": ["negotiation","negotiate","contract","vendor management",
"procurement","SLA negotiation","partnership agreement"],
"Instructing": ["instruct","mentor","coach","teach","train","knowledge transfer",
"pair programming","onboard","technical guidance","SME"],
"Service Orientation": ["customer service","client-facing","user support","help desk",
"service delivery","customer success","service level","support ticket"],
# ── COMPLEX PROBLEM SOLVING ────────────────────────────────────────────────
"Complex Problem Solving": ["complex problem","problem solving","debug","troubleshoot",
"root cause analysis","issue resolution","incident response","triage",
"architecture challenge","unstructured problem","ambiguity"],
# ── TECHNICAL SKILLS ──────────────────────────────────────────────────────
"Operations Analysis": ["operations analysis","business analysis","requirements analysis",
"workflow analysis","process improvement","gap analysis","cost-benefit",
"functional requirements","business requirements"],
"Technology Design": ["technology design","system design","architecture","solution design",
"technical design","UI/UX","API design","data modeling","schema design",
"design pattern","technical specification"],
"Equipment Selection": ["tool selection","hardware","infrastructure",
"tool evaluation","vendor evaluation","technology selection","build vs buy"],
"Installation": ["install","deploy","configure","setup","provisioning",
"implementation","rollout","release","CI/CD","Terraform","Helm","ArgoCD"],
"Programming": ["programming","coding","software development","software engineering",
"develop","script","Python","Java","SQL","C++","JavaScript","TypeScript",
"Scala","Go","algorithm","data structure","object-oriented","write code"],
"Operations Monitoring": ["system monitoring","infrastructure monitoring",
"network monitoring","application performance","Datadog","Prometheus",
"Grafana","CloudWatch","PagerDuty","incident management","APM"],
"Operation and Control": ["operation and control","automation","DevOps",
"orchestration","Kubernetes","Airflow","workflow automation",
"MLOps","DataOps","pipeline orchestration","job scheduling"],
"Equipment Maintenance": ["maintenance","patching","system upkeep",
"server maintenance","database maintenance","lifecycle management",
"vulnerability patching"],
"Troubleshooting": ["troubleshoot","debug","diagnose","fix bugs","resolve issues",
"issue tracking","defect","log analysis","error handling","postmortem","RCA"],
"Repairing": ["repair","restore","recovery","disaster recovery","failover",
"remediation","rollback","hotfix","business continuity","DR plan"],
"Quality Control Analysis": ["quality control","QA","QC","testing","unit test",
"integration test","code review","validation","test automation","regression",
"acceptance testing","TDD","pytest","Selenium","performance test"],
# ── SYSTEMS SKILLS ────────────────────────────────────────────────────────
"Judgment and Decision Making": ["decision making","judgment","prioritize",
"strategic decision","risk assessment","trade-off","governance",
"risk management","cost-benefit analysis"],
"Systems Analysis": ["systems analysis","enterprise architecture","data architecture",
"integration design","microservices architecture","distributed systems",
"scalability","end-to-end design","full stack","platform architecture"],
"Systems Evaluation": ["benchmark","performance testing","load test","audit",
"proof of concept","A/B testing","model evaluation","ablation","retrospective"],
# ── RESOURCE MANAGEMENT ────────────────────────────────────────────────────
"Time Management": ["time management","deadline","prioritization","scheduling",
"project management","sprint","agile","scrum","milestone","roadmap",
"backlog","Jira","Confluence"],
"Management of Financial Resources": ["budget","financial management","cost management",
"ROI","P&L","revenue","forecast","financial planning","cost optimization",
"fiscal","cost savings"],
"Management of Material Resources": ["resource allocation","inventory","asset management",
"procurement","capacity planning","cloud cost","resource optimization"],
"Management of Personnel Resources": ["people management","team management","hiring",
"performance review","talent management","workforce planning","manage team",
"supervise","org design","team building","recruitment"],
}
ONET_FAMILIES = {
"Basic Skills":
["Reading Comprehension","Active Listening","Writing","Speaking",
"Mathematics","Science","Critical Thinking","Active Learning",
"Learning Strategies","Monitoring"],
"Social Skills":
["Social Perceptiveness","Coordination","Persuasion",
"Negotiation","Instructing","Service Orientation"],
"Complex Problem Solving":["Complex Problem Solving"],
"Technical Skills":
["Operations Analysis","Technology Design","Equipment Selection",
"Installation","Programming","Operations Monitoring",
"Operation and Control","Equipment Maintenance",
"Troubleshooting","Repairing","Quality Control Analysis"],
"Systems Skills":
["Judgment and Decision Making","Systems Analysis","Systems Evaluation"],
"Resource Management":
["Time Management","Management of Financial Resources",
"Management of Material Resources","Management of Personnel Resources"],
}
ONET_SKILL_TO_FAMILY = {sk: fam for fam, sks in ONET_FAMILIES.items() for sk in sks}


def map_onet_skills(title='', description='', existing_skills=None):
    corpus = (str(title) + ' ' + str(description) + ' ' + ' '.join(existing_skills or
    [])).lower()
    return [sk for sk, triggers in ONET_SKILLS_MAP.items()
            if any(t.lower() in corpus for t in triggers)]

def onet_family_profile(onet_skills):
    p = {f: 0 for f in ONET_FAMILIES}
    for sk in onet_skills:
        fam = ONET_SKILL_TO_FAMILY.get(sk)
        if fam: p[fam] += 1
    return p
print(f"O*NET {ONET_VERSION} — {len(ONET_SKILLS_MAP)} skill elements across {len(ONET_FAMILIES)} families.")

O*NET O*NET 24.2 — 35 skill elements across 6 families.


--## Section 4 — USAJobs API: Maximum-Depth Federal Collection
**Endpoint:** `https://data.usajobs.gov/api/search`
**Auth:** `Authorization-Key` header + `User-Agent` (registered email)
**Expected runtime:** ~18–25 min for full 10-page depth across all 37 keywords.

In [ ]:
USAJOBS_ENDPOINT = 'https://data.usajobs.gov/api/search'

def usajobs_get_page(keyword, page=1):
    """Fetch one page of USAJobs results with up to 3 retries + exponential
    back-off on 429."""
    if not (USAJOBS_KEY and USAJOBS_EMAIL):
        return []
    headers = {
        'Host': 'data.usajobs.gov',
        'User-Agent': USAJOBS_EMAIL,
        'Authorization-Key': USAJOBS_KEY,
    }
    params = {
        'Keyword': keyword,
        'ResultsPerPage': USAJOBS_PAGE_SIZE,
        'Page': page,
        'CountrySubDivisionCode': 'US',
    }
    for attempt in range(3):
        try:
            r = requests.get(USAJOBS_ENDPOINT, headers=headers, params=params,
                            timeout=30)
            r.raise_for_status()
            return r.json().get('SearchResult', {}).get('SearchResultItems', [])
        except requests.exceptions.HTTPError as e:
            if r.status_code == 429:
                wait = (2 ** attempt) * 10
                print(f'[USAJobs] Rate limited — waiting {wait}s')
                time.sleep(wait)
            else:
                print(f'[USAJobs] HTTP {r.status_code} "{keyword}" p{page}')
                return []
        except Exception as e:
            print(f'[USAJobs] Error attempt {attempt+1}: {e}')
            time.sleep(5)
    return []

def parse_usajobs_item(item, keyword, domain):
    mv = item.get('MatchedObjectDescriptor', {})
    title = mv.get('PositionTitle', '')
    org = mv.get('OrganizationName', 'Federal Agency')
    locs = mv.get('PositionLocation', [])
    loc_raw = locs[0].get('LocationName', '') if locs else ''
    qual = mv.get('QualificationSummary', '')
    ua = mv.get('UserArea', {}).get('Details', {})
    duties = ua.get('MajorDuties', [])
    desc = clean_html(qual + ' ' + (' '.join(duties) if isinstance(duties, list)
                                   else str(duties)))
    pub_date = mv.get('PublicationStartDate', '')[:10]
    close_dt = mv.get('ApplicationCloseDate', '')[:10]
    url = mv.get('PositionURI', '')
    rem = mv.get('PositionRemuneration', [])
    salary = ''
    if rem:
        lo = rem[0].get('MinimumRange', ''); hi = rem[0].get('MaximumRange', '')
        iv = rem[0].get('RateIntervalCode', '')
        salary = f'${lo}-${hi} {iv}'.strip()
    sched = mv.get('PositionSchedule', [])
    job_type = sched[0].get('Name', '') if sched else ''
    grades = mv.get('JobGrade', [])
    grade = ', '.join(g.get('Code', '') for g in grades)
    score, matched, tier = score_relevance(title, desc, org)
    tech = extract_skills(title, desc)
    onet = map_onet_skills(title, desc, tech)
    return {
        'hash_id': make_hash(title, org, loc_raw),
        'job_title': title,
        'company': org,
        'location': normalize_location(loc_raw),
        'required_skills': tech,
        'technical_skills': tech,
        'onet_skills': onet,
        'domain': domain,
        'era': classify_era(pub_date),
        'publication_date': pub_date,
        'application_close': close_dt,
        'salary': salary,
        'job_type': job_type,
        'grade': grade,
        'job_url': url,
        'description': desc[:600],
        'relevance_score': score,
        'matched_terms': '|'.join(matched),
        'relevance_tier': tier,
        'source': 'USAJobs',
        'search_keyword': keyword,
    }

def collect_usajobs(queries=None, max_pages=None, min_score=None):
    queries = queries or USAJOBS_QUERIES
    max_pages = USAJOBS_MAX_PAGES if max_pages is None else max_pages
    min_score = MIN_SCORE if min_score is None else min_score
    if not (USAJOBS_KEY and USAJOBS_EMAIL):
        print('USAJobs credentials missing — skipping.')
        return []
    all_records, seen = [], set()
    for domain, keywords in queries.items():
        for keyword in tqdm(keywords, desc=f'USAJobs [{domain}]'):
            for page in range(1, max_pages + 1):
                items = usajobs_get_page(keyword, page)
                if not items: break
                for item in items:
                    mv = item.get('MatchedObjectDescriptor', {})
                    t = mv.get('PositionTitle', '')
                    org = mv.get('OrganizationName', '')
                    locs = mv.get('PositionLocation', [])
                    loc = locs[0].get('LocationName', '') if locs else ''
                    hid = make_hash(t, org, loc)
                    if hid in seen: continue
                    qs, _, _ = score_relevance(t, '', org)
                    if qs == 0: continue
                    rec = parse_usajobs_item(item, keyword, domain)
                    if rec['relevance_score'] >= min_score:
                        seen.add(hid)
                        all_records.append(rec)
                time.sleep(USAJOBS_DELAY)
    print(f'USAJobs total: {len(all_records)} records')
    return all_records
print("USAJobs collector ready.")

USAJobs collector ready.


In [ ]:
print('Collecting from USAJobs...')
print('=' * 60)

try:
    usajobs_records = collect_usajobs()
    if usajobs_records:
        df_usa = pd.DataFrame(usajobs_records)
        print(f'\n Records : {len(df_usa)}')
        print(f' HIGH tier : {(df_usa.relevance_tier=="HIGH").sum()}')
        print(f' Current (>=2026): {(df_usa.era=="current").sum()}')
        print('\nDomain breakdown:')
        print(df_usa.domain.value_counts().to_string())
        all_onet_u = list(chain.from_iterable(df_usa.onet_skills.dropna()))
        print('\nTop 8 O*NET skills (USAJobs):')
        for sk, n in Counter(all_onet_u).most_common(8):
            print(f' {sk:<42}: {n}')
    else:
        df_usa = pd.DataFrame()
        print('No USAJobs results — verify USAJOBS_EMAIL and USAJOBS_KEY in Colab Secrets.')
except Exception as e:
    print(f"An error occurred during USAJobs collection: {e}")
    df_usa = pd.DataFrame() # Ensure df_usa is defined even on error

USAJobs [CS]:   0%|          | 0/12 [00:00<?, ?it/s]

USAJobs [MBA]:   0%|          | 0/10 [00:00<?, ?it/s]

USAJobs [DATA_ANALYTICS]:   0%|          | 0/10 [00:00<?, ?it/s]

USAJobs [AI]:   0%|          | 0/15 [00:00<?, ?it/s]

USAJobs total: 1008 records

 Records : 1008
 HIGH tier : 416
 Current (>=2026): 968

Domain breakdown:
domain
MBA               459
CS                394
DATA_ANALYTICS    107
AI                 48

Top 8 O*NET skills (USAJobs):
 Programming                               : 989
 Instructing                               : 889
 Critical Thinking                         : 879
 Monitoring                                : 756
 Writing                                   : 707
 Coordination                              : 503
 Science                                   : 446
 Installation                              : 429


--## Section 5 — Jooble API: Maximum-Depth Commercial Collection
**Endpoint:** `https://jooble.org/api/{key}` (POST)
**Quota:** 500 requests/day free tier
**Expected runtime:** ~8–12 min for full 5-page depth across all 33 keywords.

In [ ]:
def jooble_get_page(keyword, page=1, location='United States'):
    if not JOOBLE_KEY: return []
    url = f'https://jooble.org/api/{JOOBLE_KEY}'
    payload = {'keywords': keyword, 'location': location, 'page': page}
    for attempt in range(3):
        try:
            r = requests.post(url, json=payload, timeout=30)
            r.raise_for_status()
            return r.json().get('jobs', [])
        except requests.exceptions.HTTPError:
            print(f'[Jooble] HTTP {r.status_code} "{keyword}" p{page}')
            return []
        except Exception as e:
            print(f'[Jooble] Error attempt {attempt+1}: {e}')
            time.sleep(5)
    return []

def parse_jooble_job(job, keyword, domain):
    title = job.get('title', '')
    company = job.get('company', 'Unknown Company')

    loc_raw = job.get('location', '')
    snippet = clean_html(job.get('snippet', '') or job.get('description', ''))
    pub_date = str(job.get('updated', ''))[:10]
    score, matched, tier = score_relevance(title, snippet, company)
    tech = extract_skills(title, snippet)
    onet = map_onet_skills(title, snippet, tech)
    return {
        'hash_id': make_hash(title, company, loc_raw),
        'job_title': title,
        'company': company,
        'location': normalize_location(loc_raw),
        'required_skills': tech,
        'technical_skills': tech,
        'onet_skills': onet,
        'domain': domain,
        'era': classify_era(pub_date),
        'publication_date': pub_date,
        'application_close': '',
        'salary': job.get('salary', ''),
        'job_type': job.get('type', ''),
        'grade': '',
        'job_url': job.get('link', ''),
        'description': snippet[:600],
        'relevance_score': score,
        'matched_terms': '|'.join(matched),
        'relevance_tier': tier,
        'source': 'Jooble',
        'search_keyword': keyword,
    }

def collect_jooble(queries=None, max_pages=None, min_score=None):
    queries = queries or JOOBLE_QUERIES
    max_pages = JOOBLE_MAX_PAGES if max_pages is None else max_pages
    min_score = MIN_SCORE if min_score is None else min_score
    if not JOOBLE_KEY:
        print('JOOBLE_KEY missing — skipping.')
        return []
    all_records, seen_fp = [], set()
    for domain, keywords in queries.items():
        for keyword in tqdm(keywords, desc=f'Jooble [{domain}]'):
            for page in range(1, max_pages + 1):
                jobs = jooble_get_page(keyword, page=page)
                if not jobs: break
                for job in jobs:
                    fp = f"{job.get('title','').lower()}|{job.get('company','').lower()}|{job.get('location','').lower()}"
                    if fp in seen_fp: continue
                    seen_fp.add(fp)

                    rec = parse_jooble_job(job, keyword, domain)
                    if rec['relevance_score'] >= min_score:
                        all_records.append(rec)
                time.sleep(JOOBLE_DELAY)
    print(f'Jooble total: {len(all_records)} records')
    return all_records
print("Jooble collector ready.")

Jooble collector ready.


In [ ]:
print('Collecting from Jooble...')
print('=' * 60)
jooble_records = collect_jooble()
if jooble_records:
    df_jooble = pd.DataFrame(jooble_records)
    print(f'\n Records : {len(df_jooble)}')
    print(f' HIGH tier : {(df_jooble.relevance_tier=="HIGH").sum()}')
    print(f' Current (>=2026): {(df_jooble.era=="current").sum()}')
    print('\nDomain breakdown:')
    print(df_jooble.domain.value_counts().to_string())
    all_onet_j = list(chain.from_iterable(df_jooble.onet_skills.dropna()))
    print('\nTop 8 O*NET skills (Jooble):')
    for sk, n in Counter(all_onet_j).most_common(8):
        print(f' {sk:<42}: {n}')
else:
    df_jooble = pd.DataFrame()
    print('No Jooble results — verify JOOBLE_KEY in Colab Secrets.')

Jooble [CS]:   0%|          | 0/11 [00:00<?, ?it/s]

Jooble [MBA]:   0%|          | 0/9 [00:00<?, ?it/s]

Jooble [DATA_ANALYTICS]:   0%|          | 0/9 [00:00<?, ?it/s]

Jooble [AI]:   0%|          | 0/13 [00:00<?, ?it/s]

Jooble total: 5845 records

 Records : 5845
 HIGH tier : 4152
 Current (>=2026): 5840

Domain breakdown:
domain
AI                1830
CS                1597
MBA               1259
DATA_ANALYTICS    1159

Top 8 O*NET skills (Jooble):
 Programming                               : 2781
 Operation and Control                     : 734
 Critical Thinking                         : 605
 Installation                              : 562
 Writing                                   : 523
 Science                                   : 465
 Monitoring                                : 460
 Equipment Selection                       : 385


In [ ]:
def build_master_json(df: pd.DataFrame) -> dict:
    collected_at = datetime.utcnow().isoformat() + 'Z'
    postings = []
    for _, row in df.iterrows():
        tech = row.get('required_skills', [])
        onet = row.get('onet_skills', [])

        # Ensure tech and onet are lists, parsing if they are strings
        if isinstance(tech, str):
            try:
                tech = json.loads(tech)
            except:
                tech = [s.strip() for s in tech.strip('[]').split(',') if s.strip()]
        if not isinstance(tech, list):
            tech = []

        if isinstance(onet, str):
            try:
                onet = json.loads(onet)
            except:
                onet = [s.strip() for s in onet.strip('[]').split(',') if s.strip()]
        if not isinstance(onet, list):
            onet = []

        postings.append({
            'id': int(row.get('id', 0)),
            'title': str(row.get('job_title', '')),
            'company': str(row.get('company', '')),
            'location': str(row.get('location', '')),
            'required_skills': tech,
            'technical_skills': tech,
            'onet_skills': onet,
            'onet_family_profile': onet_family_profile(onet),
            'domain': str(row.get('domain', '')),
            'era': str(row.get('era', '')),
            'posted_date': str(row.get('publication_date', '')),
            'application_close': str(row.get('application_close', '')),
            'salary': str(row.get('salary', '')),
            'job_type': str(row.get('job_type', '')),
            'grade': str(row.get('grade', '')),
            'job_url': str(row.get('job_url', '')),
            'description': str(row.get('description', '')),
            'source': str(row.get('source', '')),
            'search_keyword': str(row.get('search_keyword', '')),
            'relevance_score': int(row.get('relevance_score', 0)),
            'matched_terms': str(row.get('matched_terms', '')),
            'relevance_tier': str(row.get('relevance_tier', '')),
            'hash_id': str(row.get('hash_id', '')),
            'collected_at': collected_at,
        })
    all_onet = list(chain.from_iterable(p['onet_skills'] for p in postings))
    onet_freq = dict(Counter(all_onet).most_common())
    fam_totals = {f: sum(onet_freq.get(sk, 0) for sk in sks)
                  for f, sks in ONET_FAMILIES.items()}
    return {
        'metadata': {
            'title': 'CS / MBA / Data Analytics / AI — MASTER Job Postings',
            'researcher': 'Rickey Patel — Penn State Great Valley D.Eng.',
            'generated_at': collected_at,
            'run_timestamp': RUN_TS,
            'total_records': len(postings),
            'current_records': sum(1 for p in postings if p['era'] == 'current'),
            'historical_records': sum(1 for p in postings if p['era'] == 'historical'),
            'current_epoch': CURRENT_EPOCH.isoformat(),
            'domains': ['CS', 'MBA', 'DATA_ANALYTICS', 'AI'],
            'sources': ['USAJobs', 'Jooble'],
            'schema_version': '3.1',
            'onet_version': ONET_VERSION,
            'onet_top_skills': dict(Counter(all_onet).most_common(15)),
            'onet_family_totals': fam_totals,
            'merge_compatible': True,
        },
        'postings': postings,
    }


frames = [df for df in [
    df_usa if 'df_usa' in dir() and not df_usa.empty else None,
    df_jooble if 'df_jooble' in dir() and not df_jooble.empty else None,
] if df is not None]
if frames:
    df_all = pd.concat(frames, ignore_index=True)
    raw_n = len(df_all)
    df_all = (df_all
              .sort_values('relevance_score', ascending=False)
              .drop_duplicates(subset='hash_id', keep='first')
              .reset_index(drop=True))
    df_all.insert(0, 'id', range(1, len(df_all) + 1))
    master = build_master_json(df_all)
    print(f"Merged: {raw_n} raw → {len(df_all)} unique ({raw_n - len(df_all)} duplicates removed)")
    print(f" Current (≥2026) : {(df_all.era=='current').sum()}")
    print(f" Historical      : {(df_all.era=='historical').sum()}")
    print(f" HIGH tier       : {(df_all.relevance_tier=='HIGH').sum()}")
    print(f" MEDIUM tier     : {(df_all.relevance_tier=='MEDIUM').sum()}")
    print(f" Mean score      : {df_all.relevance_score.mean():.2f}")
    print('\nDomain breakdown:')
    print(df_all.domain.value_counts().to_string())
    print('\nO*NET top 10 (merged):')
    for sk, n in list(master['metadata']['onet_top_skills'].items())[:10]:
        print(f' {sk:<42}: {n:>5} [{ONET_SKILL_TO_FAMILY.get(sk,"")}]')
else:
    print('No data — run Sections 4 and 5 first.')

Merged: 6853 raw → 6853 unique (0 duplicates removed)
 Current (≥2026) : 6808
 Historical      : 45
 HIGH tier       : 4568
 MEDIUM tier     : 313
 Mean score      : 5.39

Domain breakdown:
domain
CS                1991
AI                1878
MBA               1718
DATA_ANALYTICS    1266

O*NET top 10 (merged):
 Programming                               :  3770 [Technical Skills]
 Critical Thinking                         :  1484 [Basic Skills]
 Instructing                               :  1258 [Social Skills]
 Writing                                   :  1230 [Basic Skills]
 Monitoring                                :  1216 [Basic Skills]
 Installation                              :   991 [Technical Skills]
 Science                                   :   911 [Basic Skills]
 Operation and Control                     :   815 [Technical Skills]
 Coordination                              :   798 [Social Skills]
 Negotiation                               :   711 [Social Skills]


--## Section 7 — Export → Google Drive
Writes three files to `DRIVE_OUTPUTS` using the mounted Drive path (`shutil.copy2`).
No additional OAuth or service-account credentials are needed.
| File | Contents |
|---|---|
| `MASTER_*.json` | Full dataset with `onet_skills` + `onet_family_profile` |
| `MASTER_*.csv` | Flat CSV, skill lists pipe-delimited |
| `MASTER_*.xlsx` | 8 sheets including O\\*NET frequency + family pivot |

In [ ]:
if 'master' not in dir() or not master:
    print('No master — run Sections 4–6 first.')
else:
    # ── JSON ──────────────────────────────────────────────────────────────────
    loc_json = os.path.join(LOCAL_WORK, MASTER_JSON)

    with open(loc_json, 'w', encoding='utf-8') as f:
        json.dump(master, f, indent=2, ensure_ascii=False)
    save_to_drive(loc_json, DRIVE_OUTPUTS, MASTER_JSON)
    # ── CSV ───────────────────────────────────────────────────────────────────
    df_csv = df_all.copy()
    for col in ['required_skills', 'technical_skills', 'onet_skills']:
        if col in df_csv.columns:
            df_csv[col] = df_csv[col].apply(
                lambda x: '|'.join(x) if isinstance(x, list) else str(x))
    loc_csv = os.path.join(LOCAL_WORK, MASTER_CSV)
    df_csv.to_csv(loc_csv, index=False, encoding='utf-8-sig')
    save_to_drive(loc_csv, DRIVE_OUTPUTS, MASTER_CSV)
    # ── XLSX — 8 sheets ───────────────────────────────────────────────────────
    df_cur = df_csv[df_csv.era == 'current'].copy()
    df_his = df_csv[df_csv.era == 'historical'].copy()
    df_hm = df_csv[df_csv.relevance_tier.isin(['HIGH', 'MEDIUM'])].copy()
    df_ai = df_csv[df_csv.domain == 'AI'].copy()
    df_csdf = df_csv[df_csv.domain == 'CS'].copy()
    all_onet_flat = list(chain.from_iterable(
        (x if isinstance(x, list) else []) for x in df_all.get('onet_skills', [])))
    onet_freq_df = pd.DataFrame(Counter(all_onet_flat).most_common(),
                                columns=['onet_skill', 'frequency'])
    onet_freq_df['family'] = onet_freq_df['onet_skill'].map(ONET_SKILL_TO_FAMILY)
    onet_freq_df['pct_postings'] = (onet_freq_df['frequency'] / max(len(df_all), 1) *
                                    100).round(1)
    family_df = pd.DataFrame([
        (f, sum(Counter(all_onet_flat).get(sk, 0) for sk in sks))
        for f, sks in ONET_FAMILIES.items()
    ], columns=['onet_family', 'total_mentions'])
    comp = pd.DataFrame({
        'Metric': ['Total','Current','Historical','HIGH','MEDIUM','LOW','Avg Score'],
        'USAJobs': [
            len(df_csv[df_csv.source=='USAJobs']),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.era=='current')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.era=='historical')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.relevance_tier=='HIGH')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.relevance_tier=='MEDIUM')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.relevance_tier=='LOW')]),
            round(df_csv[df_csv.source=='USAJobs'].relevance_score.mean(), 2),
        ],
        'Jooble': [
            len(df_csv[df_csv.source=='Jooble']),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.era=='current')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.era=='historical')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.relevance_tier=='HIGH')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.relevance_tier=='MEDIUM')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.relevance_tier=='LOW')]),
            round(df_csv[df_csv.source=='Jooble'].relevance_score.mean(), 2),
        ],
        'COMBINED': [
            len(df_csv),
            (df_csv.era=='current').sum(), (df_csv.era=='historical').sum(),
            (df_csv.relevance_tier=='HIGH').sum(),
            (df_csv.relevance_tier=='MEDIUM').sum(),
            (df_csv.relevance_tier=='LOW').sum(),
            round(df_csv.relevance_score.mean(), 2),
        ],
    })
    loc_xlsx = os.path.join(LOCAL_WORK, MASTER_XLSX)
    with pd.ExcelWriter(loc_xlsx, engine='openpyxl') as w:
        df_csv.to_excel(w,
                            sheet_name='All Postings',
                            index=False)
        df_cur.to_excel(w,
                            sheet_name='Current (2026+)',
                            index=False)
        df_his.to_excel(w,
                            sheet_name='Historical (<2026)',
                            index=False)
        df_hm.to_excel(w,
                            sheet_name='HIGH + MEDIUM Tier',
                            index=False)
        df_ai.to_excel(w,
                            sheet_name='AI Domain',
                            index=False)
        df_csdf.to_excel(w,
                            sheet_name='CS Domain',
                            index=False)
        onet_freq_df.to_excel(w, sheet_name='ONET Skill Frequency', index=False)
        family_df.to_excel(w,
                            sheet_name='ONET Family Summary',
                            index=False)
        comp.to_excel(w,
                            sheet_name='Source Comparison',
                            index=False)
    save_to_drive(loc_xlsx, DRIVE_OUTPUTS, MASTER_XLSX)
    # ── Run log ───────────────────────────────────────────────────────────────
    write_run_log(df_all)
    print('\nAll files saved to Drive:')
    for f in sorted(os.listdir(DRIVE_OUTPUTS)):
        kb = os.path.getsize(os.path.join(DRIVE_OUTPUTS, f)) / 1024
        print(f' {f:<65} ({kb:.1f} KB)')
    sample = \
        [{'id':p['id'],'title':p['title'],'required_skills':p['required_skills'][:3],
          'onet_skills':p['onet_skills'][:4]}
         for p in master['postings'] if p['era']=='current'][:3]
    print('\nSample current postings:')
    print(json.dumps(sample, indent=2))

MASTER_20260624_1420.json (10132.8 KB) → /content/drive/MyDrive/Research/JobPostings/outputs ✅
MASTER_20260624_1420.csv (4252.6 KB) → /content/drive/MyDrive/Research/JobPostings/outputs ✅
MASTER_20260624_1420.xlsx (6122.4 KB) → /content/drive/MyDrive/Research/JobPostings/outputs ✅

All files saved to Drive:
 DRY_RUN_TEST_20260620_2057.json                                   (127.4 KB)
 DRY_RUN_TEST_20260621_1159.json                                   (128.3 KB)
 DRY_RUN_TEST_20260623_1840.json                                   (293.3 KB)
 IT_jobs_google_jobs_AllRecords.json                               (12148.2 KB)
 MASTER 20260621 MERGED.json                                       (5030.8 KB)
 MASTER_20260620_1350.csv                                          (1970.4 KB)
 MASTER_20260620_1350.json                                         (3560.5 KB)
 MASTER_20260620_1350.xlsx                                         (1432.5 KB)
 MASTER_20260620_1621.csv                                    

--## Section 8 — Automation Backends
Three production-ready options. Pick one based on your infrastructure.
All three write output files to the same `DRIVE_OUTPUTS` path — no code changes
needed.

--### Option A — Google Apps Script (Simplest)
**How it works:** A JS snippet in your Google account calls the Colab execution API
every Monday at 6 AM. Zero extra accounts, zero cost, entirely inside Google.
**Known limitation:** The Colab execute API is internal/unofficial. If your notebook
isn't found on Drive or the trigger gets an HTTP 4xx, re-save the notebook to Drive
and verify the file ID is correct.
#### Step 1 — Get the notebook file ID
Open the notebook in Colab. The file ID is the long string in the URL:
```
https://colab.research.google.com/drive/1ABC...XYZ
^^^^^^^^^^^ ← copy this
```
#### Step 2 — Create the Apps Script project
1. Go to **[script.google.com](https://script.google.com)** → **New project**
2. Paste the code below; replace `YOUR_NOTEBOOK_FILE_ID`
3. Save → Run → `weeklyJobCollection` (one-time permission grant)
```javascript
// ── Google Apps Script: Weekly Job Collection ────────────────────────────────
const NOTEBOOK_FILE_ID = 'YOUR_NOTEBOOK_FILE_ID'; // ← paste here
const NOTIFY_EMAIL
= Session.getActiveUser().getEmail();
function executeNotebook() {
const token = ScriptApp.getOAuthToken();
const url
=
`https://colab.research.google.com/v2/drives/drive/files/${NOTEBOOK_FILE_ID}:execute`;
const opts = {
method:
'post',
headers:
{ Authorization: `Bearer ${token}`, 'Content-Type':
'application/json' },
payload:
JSON.stringify({ accelerator_type: 'NONE' }),
muteHttpExceptions: true,
};
const resp = UrlFetchApp.fetch(url, opts);
const code = resp.getResponseCode();
Logger.log(`HTTP ${code} — ${resp.getContentText().substring(0, 200)}`);
return code;
}
function weeklyJobCollection() {
Logger.log('Run started: ' + new Date());
const code = executeNotebook();

✅
❌

const subject = code >= 200 && code < 300
? '
Job Pipeline Triggered'
: `
Job Pipeline FAILED (HTTP ${code})`;
const body = code >= 200 && code < 300
? `Pipeline triggered at ${new Date()}. Check Drive/Research/JobPostings/outputs/
in ~35 min.`
: `Trigger failed (HTTP ${code}). Verify notebook file ID and that notebook is
saved in Drive.`;
GmailApp.sendEmail(NOTIFY_EMAIL, subject, body);
}
// Run this once to create the Monday 6 AM trigger programmatically
function createWeeklyTrigger() {
ScriptApp.getProjectTriggers()
.filter(t => t.getHandlerFunction() === 'weeklyJobCollection')
.forEach(t => ScriptApp.deleteTrigger(t));
ScriptApp.newTrigger('weeklyJobCollection')
.timeBased().onWeekDay(ScriptApp.WeekDay.MONDAY).atHour(6).create();
Logger.log('Trigger created: every Monday at 6 AM.');
}
```
#### Step 3 — Add the trigger
Run `createWeeklyTrigger()` once from the Apps Script editor, **or** add it manually:
Clock icon → Add Trigger → `weeklyJobCollection` → Time-driven → Week → Monday → 6–7
AM.
#### Troubleshooting Option A
| Error | Fix |
|---|---|
| `HTTP 404` on execute | Re-save notebook: File → Save a copy in Drive. Re-copy file
ID from new URL. |
| `HTTP 403` / auth failure | Run `weeklyJobCollection` manually first to re-grant
permissions. |
| Files not appearing in Drive | Verify `DRIVE_BASE` path in Section 2 matches your
actual Drive folder. |
| Colab not finishing | Runtime may have been recycled. Re-open notebook, run once
manually, then retry. |
--### Option B — GitHub Actions (Most Reliable Free)
**How it works:** A YAML workflow runs on a GitHub-hosted runner on a cron schedule.
Papermill executes the notebook headlessly. Outputs upload to Drive via a service
account.
**Cost:** Free (2,000 min/month public repo; 500 min/month private). ~30 min per run.

#### Step 1 — Repository setup
```
your-repo/
├── CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb
├── .github/
│
└── workflows/
│
└── weekly_job_collection.yml
└── upload_to_drive.py
```

← this notebook

← paste YAML below
← paste script below

#### Step 2 — Add GitHub repository secrets
`Settings → Secrets and variables → Actions → New repository secret`
| Secret | Value |
|---|---|
| `USAJOBS_EMAIL` | Your USAJobs registered email |
| `USAJOBS_KEY` | USAJobs API key |
| `JOOBLE_KEY` | Jooble API key |
| `GDRIVE_SERVICE_ACCOUNT_JSON` | Contents of GCP service account JSON (see Step 4) |
| `GDRIVE_FOLDER_ID` | ID of the Drive `outputs` folder (from its URL) |
#### Step 3 — `.github/workflows/weekly_job_collection.yml`
```yaml
name: Weekly Job Collection
on:
schedule:
- cron: '0 6 * * 1'
workflow_dispatch:

# Every Monday 06:00 UTC
# Manual trigger from GitHub UI

jobs:
collect:
runs-on: ubuntu-latest
timeout-minutes: 75
steps:
- uses: actions/checkout@v4
- uses: actions/setup-python@v5
with: { python-version: '3.11' }
- name: Install dependencies
run: |
pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \
jupyter nbconvert google-auth google-auth-oauthlib \
google-api-python-client

- name: Execute notebook
env:
USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
USAJOBS_KEY:
${{ secrets.USAJOBS_KEY }}
JOOBLE_KEY:
${{ secrets.JOOBLE_KEY }}
run: |
mkdir -p /tmp/job_outputs /tmp/job_work
papermill CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb \
/tmp/executed_notebook.ipynb \
--no-progress-bar \
-p DRIVE_BASE /tmp/job_outputs
- name: Upload outputs to Google Drive
env:
GDRIVE_SERVICE_ACCOUNT_JSON: ${{ secrets.GDRIVE_SERVICE_ACCOUNT_JSON }}
GDRIVE_FOLDER_ID:
${{ secrets.GDRIVE_FOLDER_ID }}
run: python upload_to_drive.py
- uses: actions/upload-artifact@v4
if: always()
with:
name: executed-notebook-${{ github.run_id }}
path: /tmp/executed_notebook.ipynb
retention-days: 14
```
#### Step 4 — `upload_to_drive.py`
```python
import os, json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.service_account import Credentials
sa_info = json.loads(os.environ['GDRIVE_SERVICE_ACCOUNT_JSON'])
creds
= Credentials.from_service_account_info(
sa_info, scopes=['https://www.googleapis.com/auth/drive.file'])
service = build('drive', 'v3', credentials=creds)
folder = os.environ['GDRIVE_FOLDER_ID']
output_dir = '/tmp/job_outputs/outputs'
if not os.path.isdir(output_dir):
output_dir = '/tmp/job_outputs'
for fname in os.listdir(output_dir):
if not fname.startswith('MASTER_'):
continue
fpath = os.path.join(output_dir, fname)
mime = ('application/json' if fname.endswith('.json') else

'text/csv'
if fname.endswith('.csv') else
'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
meta = {'name': fname, 'parents': [folder]}
media = MediaFileUpload(fpath, mimetype=mime, resumable=True)
f
= service.files().create(body=meta, media_body=media, fields='id').execute()
print(f'Uploaded {fname} → Drive ID {f.get("id")}')
```
#### Step 5 — Create the GCP service account (one time)
```bash
# 1. Create service account
gcloud iam service-accounts create github-pipeline \
--display-name="GitHub Actions Job Pipeline" \
--project=YOUR_PROJECT_ID
# 2. Download JSON key
gcloud iam service-accounts keys create sa_key.json \
--iam-account=github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
# 3. Share your Drive outputs folder with the service account email:
#
Open Drive folder → Share → paste
github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
#
→ Editor access
# 4. Paste the contents of sa_key.json into the GDRIVE_SERVICE_ACCOUNT_JSON secret
```
#### Troubleshooting Option B
| Error | Fix |
|---|---|
| `google.colab` import fails | The notebook detects Colab automatically. The
`_get_secret()` helper in Section 0 falls back to `os.environ` on non-Colab runtimes —
no code change needed. |
| Drive upload `insufficientPermissions` | Verify the service account email has
**Editor** access on the Drive folder (not just the file). |
| `papermill` parameter not injected | Ensure Section 2 uses
`globals().get('DRIVE_BASE', …)` — it does by default. |
| Workflow timeout | Increase `timeout-minutes` to 90. Full run is typically 35–45
min. |
--### Option C — GCP Cloud Scheduler + Cloud Run (Production)
**How it works:** A containerised Python script runs on Cloud Run (serverless),
triggered
weekly by Cloud Scheduler. Outputs go to GCS; an optional Drive sync step writes them
to Google Drive as well.

**Cost:** Cloud Run ~$0.20/run · Cloud Scheduler $0.10/job/month → **~$1/month
total**.
#### Architecture
```
Cloud Scheduler (cron) ──HTTP POST──► Cloud Run Job
│
Executes pipeline.py
│
┌────────────┴────────────┐
▼
▼
GCS bucket
Google Drive
gs://PROJECT-outputs/
(optional sync)
```
#### Step 1 — Prepare the standalone script
```bash
# Convert notebook to script (run locally or in Colab)
jupyter nbconvert --to script CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb \
--output pipeline
# Remove all google.colab imports — replace with os.environ.get() (already done
# by the _get_secret() helper in Section 0)
# Remove the colab_drive.mount() call — replace with local /tmp paths
```
Wrap the script in an `if __name__ == '__main__':` guard and change `DRIVE_BASE`
to `/tmp/job_outputs`. Add the GCS upload block below at the end.
#### Step 2 — GCS upload block (add to end of pipeline.py)
```python
from google.cloud import storage as gcs
def upload_to_gcs(local_path, bucket_name, blob_name):
client = gcs.Client()
bucket = client.bucket(bucket_name)
blob
= bucket.blob(blob_name)
blob.upload_from_filename(local_path)
print(f'GCS: gs://{bucket_name}/{blob_name}')
BUCKET = os.environ.get('GCS_BUCKET', 'YOUR_PROJECT_ID-outputs')
for fname in [MASTER_JSON, MASTER_CSV, MASTER_XLSX]:
local = os.path.join('/tmp/job_outputs', fname)
if os.path.exists(local):
upload_to_gcs(local, BUCKET, fname)
```

#### Step 3 — Dockerfile
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY pipeline.py .
CMD ["python", "pipeline.py"]
```
#### Step 4 — requirements.txt
```
pandas==2.2.2
openpyxl==3.1.2
requests==2.32.3
beautifulsoup4==4.12.3
tqdm==4.66.4
google-cloud-storage==2.17.0
```
#### Step 5 — Deploy
```bash
PROJECT_ID=your-project-id
REGION=us-central1
IMAGE=gcr.io/${PROJECT_ID}/job-pipeline
# Build and push
gcloud builds submit --tag ${IMAGE} --project=${PROJECT_ID}
# Create GCS bucket
gcloud storage buckets create gs://${PROJECT_ID}-outputs \
--location=${REGION} --project=${PROJECT_ID}
# Store secrets in Secret Manager
echo -n "$USAJOBS_EMAIL" | gcloud secrets create USAJOBS_EMAIL --data-file=--project=${PROJECT_ID}
echo -n "$USAJOBS_KEY"
| gcloud secrets create USAJOBS_KEY
--data-file=--project=${PROJECT_ID}
echo -n "$JOOBLE_KEY"
| gcloud secrets create JOOBLE_KEY
--data-file=--project=${PROJECT_ID}
# Create Cloud Run Job
gcloud run jobs create job-pipeline \
--image=${IMAGE} --region=${REGION} \
--memory=2Gi --cpu=1 --task-timeout=3600 \

--set-secrets=USAJOBS_EMAIL=USAJOBS_EMAIL:latest,USAJOBS_KEY=USAJOBS_KEY:latest,JOOBLE
_KEY=JOOBLE_KEY:latest \
--set-env-vars=GCS_BUCKET=${PROJECT_ID}-outputs \
--service-account=pipeline-sa@${PROJECT_ID}.iam.gserviceaccount.com \
--project=${PROJECT_ID}
# Create Cloud Scheduler weekly trigger
gcloud scheduler jobs create http weekly-job-collection \
--location=${REGION} \
--schedule="0 6 * * 1" \
--time-zone="America/New_York" \
--uri="https://${REGION}-run.googleapis.com/apis/run.googleapis.com/v1/namespaces/${PR
OJECT_ID}/jobs/job-pipeline:run" \
--http-method=POST \
--oauth-service-account-email=pipeline-sa@${PROJECT_ID}.iam.gserviceaccount.com \
--project=${PROJECT_ID}
```
#### Step 6 — Grant IAM roles
```bash
SA=pipeline-sa@${PROJECT_ID}.iam.gserviceaccount.com
# Read secrets
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
--member=serviceAccount:${SA} --role=roles/secretmanager.secretAccessor
# Write to GCS bucket
gcloud storage buckets add-iam-policy-binding gs://${PROJECT_ID}-outputs \
--member=serviceAccount:${SA} --role=roles/storage.objectCreator
# Allow Scheduler to invoke Cloud Run
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
--member=serviceAccount:${SA} --role=roles/run.invoker
```
#### Troubleshooting Option C
| Error | Fix |
|---|---|
| `PERMISSION_DENIED` on Secret Manager | Grant `roles/secretmanager.secretAccessor`
to pipeline-sa (Step 6). |
| Container exits immediately | Check Cloud Run logs: `gcloud run jobs executions
describe <exec-id> --region=us-central1 --log-http`. Common cause: missing secrets. |
| GCS upload `403` | Grant `roles/storage.objectCreator` on the specific bucket — not
just project-level. |
| `google.colab` ImportError | Remove `from google.colab import drive` from
pipeline.py and replace `DRIVE_BASE` with `/tmp/job_outputs`. |
| Cloud Scheduler `DEADLINE_EXCEEDED` | The job is still running. Increase

`--task-timeout` to 3600 (1 hr). |

### Step-by-Step Guide: Setting Up GitHub Actions

This guide outlines the process to automate your Colab notebook using GitHub Actions, as described in 'Option B' of your notebook. This method is generally reliable for scheduled, headless execution.

#### Step 1: Repository Setup

First, you need to set up your GitHub repository with the correct file structure. This involves placing your Colab notebook and creating the necessary directories for the GitHub Actions workflow and a helper script.

Your repository should look like this:

```
your-repo/
├── CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb  ← this notebook
├── .github/
│   └── workflows/
│       └── weekly_job_collection.yml               ← paste YAML below
└── upload_to_drive.py                              ← paste script below
```

Ensure your `CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb` file (or whatever your notebook is named) is at the root of your repository.

#### Step 2: Add GitHub Repository Secrets

You'll need to store your API keys and Google Drive service account JSON as GitHub repository secrets. This keeps your sensitive information secure.

1.  Go to your GitHub repository on `github.com`.
2.  Navigate to `Settings` → `Secrets and variables` → `Actions`.
3.  Click `New repository secret` for each of the following:

    | Secret                        | Value                                                               |
    | :---------------------------- | :------------------------------------------------------------------ |
    | `USAJOBS_EMAIL`               | Your USAJobs registered email                                       |
    | `USAJOBS_KEY`                 | USAJobs API key                                                     |
    | `JOOBLE_KEY`                  | Jooble API key                                                      |
    | `GDRIVE_SERVICE_ACCOUNT_JSON` | Contents of GCP service account JSON (from Step 5)                  |
    | `GDRIVE_FOLDER_ID`            | The ID of your Google Drive `outputs` folder (from its URL)         |

#### Step 3: Create `.github/workflows/weekly_job_collection.yml`

This YAML file defines the GitHub Actions workflow. It specifies when the workflow runs, what environment it uses, and the steps to execute your notebook and upload outputs.

Create the file `weekly_job_collection.yml` inside the `.github/workflows/` directory in your repository, and paste the following content:

```yaml
name: Weekly Job Collection
on:
  schedule:
    - cron: '0 6 * * 1' # Every Monday 06:00 UTC
  workflow_dispatch: # Manual trigger from GitHub UI

jobs:
  collect:
    runs-on: ubuntu-latest
    timeout-minutes: 75
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - name: Install dependencies
        run: |
          pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \
            jupyter nbconvert google-auth google-auth-oauthlib \
            google-api-python-client
      - name: Execute notebook
        env:
          USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
          USAJOBS_KEY: ${{ secrets.USAJOBS_KEY }}
          JOOBLE_KEY: ${{ secrets.JOOBLE_KEY }}
        run: |
          mkdir -p /tmp/job_outputs /tmp/job_work
          papermill CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb \
            /tmp/executed_notebook.ipynb \
            --no-progress-bar \
            -p DRIVE_BASE /tmp/job_outputs
      - name: Upload outputs to Google Drive
        env:
          GDRIVE_SERVICE_ACCOUNT_JSON: ${{ secrets.GDRIVE_SERVICE_ACCOUNT_JSON }}
          GDRIVE_FOLDER_ID: ${{ secrets.GDRIVE_FOLDER_ID }}
        run: python upload_to_drive.py
      - uses: actions/upload-artifact@v4
        if: always()
        with:
          name: executed-notebook-${{ github.run_id }}
          path: /tmp/executed_notebook.ipynb
          retention-days: 14
```

#### Step 4: Create `upload_to_drive.py`

This Python script handles uploading the generated output files from the GitHub Actions runner to your Google Drive folder using the service account credentials.

Create the file `upload_to_drive.py` at the root of your repository (the same directory as your notebook) and paste the following content:

```python
import os, json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.service_account import Credentials

sa_info = json.loads(os.environ['GDRIVE_SERVICE_ACCOUNT_JSON'])
creds = Credentials.from_service_account_info(
    sa_info, scopes=['https://www.googleapis.com/auth/drive.file'])
service = build('drive', 'v3', credentials=creds)
folder = os.environ['GDRIVE_FOLDER_ID']

output_dir = '/tmp/job_outputs/outputs'
if not os.path.isdir(output_dir):
    output_dir = '/tmp/job_outputs'

for fname in os.listdir(output_dir):
    if not fname.startswith('MASTER_'):
        continue
    fpath = os.path.join(output_dir, fname)
    mime = ('application/json' if fname.endswith('.json') else
            'text/csv' if fname.endswith('.csv') else
            'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

    meta = {'name': fname, 'parents': [folder]}
    media = MediaFileUpload(fpath, mimetype=mime, resumable=True)
    f = service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'Uploaded {fname} → Drive ID {f.get("id")}')
```

#### Step 5: Create the GCP Service Account (One Time)

This service account will be used by GitHub Actions to authenticate with Google Drive and upload files. You need to create this once in your Google Cloud Project.

1.  **Create Service Account:** Open Google Cloud Shell or your local terminal with `gcloud` SDK configured, and run:
    ```bash
    # Replace YOUR_PROJECT_ID with your actual Google Cloud Project ID
    gcloud iam service-accounts create github-pipeline \
        --display-name="GitHub Actions Job Pipeline" \
        --project=YOUR_PROJECT_ID
    ```

2.  **Download JSON Key:** Generate a JSON key for the service account:
    ```bash
    gcloud iam service-accounts keys create sa_key.json \
        --iam-account=github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
    ```
    This will download a `sa_key.json` file. The *contents* of this file are what you'll paste into the `GDRIVE_SERVICE_ACCOUNT_JSON` GitHub secret.

3.  **Share your Drive outputs folder:**
    *   Open your Google Drive `outputs` folder in your browser.
    *   Click on "Share" (or the person icon).
    *   Paste the service account email: `github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com`
    *   Grant it **Editor** access (important for uploading files).

4.  **Paste JSON into GitHub Secret:** Copy the *entire content* of the `sa_key.json` file you downloaded and paste it as the value for the `GDRIVE_SERVICE_ACCOUNT_JSON` secret in your GitHub repository (as described in Step 2).

Once all these steps are completed, your GitHub Action workflow should be configured to run your notebook weekly and upload the results to your Google Drive!

### Workaround: GitHub Actions with Workload Identity Federation

This section replaces **Step 5: Create the GCP Service Account (One Time)** and involves modifications to **Steps 2, 3, and 4** of the standard GitHub Actions setup, as your organization's policy prevents direct service account key creation. Workload Identity Federation is the recommended and most secure approach in such scenarios.

#### Google Cloud Setup

These steps will be performed in your Google Cloud Project using the `gcloud` CLI (e.g., in Cloud Shell).

##### Step GC1: Verify/Create Service Account

You've already created the `github-pipeline` service account. We'll use this existing service account. If you hadn't created it, the command would be:

```bash
gcloud iam service-accounts create github-pipeline \
    --display-name="GitHub Actions Job Pipeline" \
    --project=project-bf62429c-050a-4ead-b92
```

##### Step GC2: Grant IAM Roles to Service Account

Ensure your `github-pipeline` service account has the necessary permissions to access Google Drive. This involves sharing your Google Drive `outputs` folder with the service account.

1.  **Open your Google Drive `outputs` folder** in your browser.
2.  Click on "Share" (or the person icon).
3.  **Paste the service account email:** `github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com` (replace with your actual project ID if different).
4.  Grant it **Editor** access (important for uploading files).

Additionally, you might want to grant it the `Storage Object Creator` role if you also plan to upload to Google Cloud Storage (though your current setup only uses Drive):

```bash
gcloud projects add-iam-policy-binding project-bf62429c-050a-4ead-b92 \
    --member="serviceAccount:github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com" \
    --role="roles/storage.objectCreator"
```

In [2]:
print('gcloud projects add-iam-policy-binding project-bf62429c-050a-4ead-b92 --member="serviceAccount:github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com" --role="roles/storage.objectCreator"')

gcloud projects add-iam-policy-binding project-bf62429c-050a-4ead-b92 --member="serviceAccount:github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com" --role="roles/storage.objectCreator"


##### Step GC3: Create Workload Identity Pool

A Workload Identity Pool manages external identities that can be granted access to your Google Cloud resources.

```bash
gcloud iam workload-identity-pools create github-pool \
    --project="project-bf62429c-050a-4ead-b92" \
    --location="global" \
    --display-name="GitHub Actions Workload Identity Pool"
```

**Note:** If you get an error that the pool already exists, you can skip this step.

In [5]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GitHub Actions Workload Identity Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GitHub Actions Workload Identity Pool"


In [6]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"


In [7]:
print('gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com"')

gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com"


In [8]:
print('gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format=\'value(projectNumber)\')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"')

gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format='value(projectNumber)')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"


In [9]:
print('gcloud iam workload-identity-pools list --project="project-bf62429c-050a-4ead-b92" --location="global"')

gcloud iam workload-identity-pools list --project="project-bf62429c-050a-4ead-b92" --location="global"


In [11]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"


After successfully running the above command and confirming the Workload Identity Pool is created, please proceed to re-run **Step GC4** and **Step GC5** from the notebook. Remember to replace `YOUR_GITHUB_ORG` and `YOUR_GITHUB_REPO` with your actual GitHub details in **Step GC5**.

In [12]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions WIF Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions WIF Pool"


##### Step GC4: Create Workload Identity Provider for GitHub

This provider establishes trust with GitHub's OIDC (OpenID Connect) provider.

```bash
gcloud iam workload-identity-pools providers create-oidc github-provider \
    --project="project-bf62429c-050a-4ead-b92" \
    --location="global" \
    --workload-identity-pool="github-pool" \
    --display-name="GitHub Actions OIDC Provider" \
    --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" \
    --issuer-uri="https://token.actions.githubusercontent.com"
```

**Note:** If you get an error that the provider already exists, you can skip this step.

##### Step GC5: Grant GitHub Identity Access to Impersonate Service Account

This is the critical step where you allow a specific GitHub repository's identity to impersonate your `github-pipeline` service account. Replace `YOUR_GITHUB_ORG` and `YOUR_GITHUB_REPO` with your actual GitHub organization/username and repository name.

```bash
gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com \
    --project="project-bf62429c-050a-4ead-b92" \
    --role="roles/iam.workloadIdentityUser" \
    --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format='value(projectNumber)')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"
```

**Important:**
*   Replace `YOUR_GITHUB_ORG` with your GitHub username or organization name.
*   Replace `YOUR_GITHUB_REPO` with the name of your GitHub repository.
*   If you're using a different branch than `main`, update `ref:refs/heads/main` accordingly (e.g., `ref:refs/heads/master`).
*   The `$(gcloud projects describe ...)` part automatically fetches your project *number*, which is required here. Ensure you're logged into the correct `gcloud` configuration.

In [17]:
print('gcloud iam workload-identity-pools providers create-oidc github-provider-v2 --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider V2" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com" --attribute-condition="attribute.repository == \'rpp5069-cpu/job-pipeline\'"')

gcloud iam workload-identity-pools providers create-oidc github-provider-v2 --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider V2" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com" --attribute-condition="attribute.repository == 'rpp5069-cpu/job-pipeline'"


In [14]:
print('gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format=\'value(projectNumber)\')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"')

gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format='value(projectNumber)')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"


In [15]:
print('gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub" --issuer-uri="https://token.actions.githubusercontent.com"')

gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub" --issuer-uri="https://token.actions.githubusercontent.com"


#### GitHub Actions Setup

Now, let's update your GitHub repository.

##### Step GA1: Update GitHub Repository Secrets

Go to your GitHub repository `Settings` → `Secrets and variables` → `Actions`.

*   **Add a new secret:**
    *   **Name:** `GCP_PROJECT_ID`
    *   **Value:** `project-bf62429c-050a-4ead-b92` (your Google Cloud Project ID)
*   **Remove the `GDRIVE_SERVICE_ACCOUNT_JSON` secret.** It is no longer needed with Workload Identity Federation.

##### Step GA2: Modify `upload_to_drive.py`

Update your `upload_to_drive.py` script to use Google's default credential mechanism, as the environment will now be authenticated via Workload Identity Federation. This means the `GDRIVE_SERVICE_ACCOUNT_JSON` environment variable will no longer be available.

Replace the contents of `upload_to_drive.py` with this updated version:

```python
import os, json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth import default # Import default credentials

# Use default credentials provided by Workload Identity Federation
creds, project = default()
service = build('drive', 'v3', credentials=creds)
folder = os.environ['GDRIVE_FOLDER_ID'] # This secret is still needed

output_dir = '/tmp/job_outputs/outputs'
if not os.path.isdir(output_dir): # Corrected: os.path() was incorrect
    output_dir = '/tmp/job_outputs'

for fname in os.listdir(output_dir):
    if not fname.startswith('MASTER_'):
        continue
    fpath = os.path.join(output_dir, fname)
    mime = ('application/json' if fname.endswith('.json') else
            'text/csv' if fname.endswith('.csv') else
            'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

    meta = {'name': fname, 'parents': [folder]}
    media = MediaFileUpload(fpath, mimetype=mime, resumable=True)
    f = service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'Uploaded {fname} → Drive ID {f.get("id")}')
```

##### Step GA3: Modify GitHub Actions Workflow YAML (`weekly_job_collection.yml`)

I have updated the `papermill` command to use the specific filename you provided: `CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb`.

```yaml
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }

      # 1. Authenticate via Workload Identity Federation
      - id: 'auth'
        uses: 'google-github-actions/auth@v2'
        with:
          workload_identity_provider: 'projects/71023542519/locations/global/workloadIdentityPools/github-pool/providers/github-provider-v2'
          service_account: 'github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com'

      # 2. Setup gcloud CLI
      - name: 'Set up Cloud SDK'
        uses: 'google-github-actions/setup-gcloud@v2'

      - name: List Repository Files (Debug)
        run: ls -R

      - name: Install dependencies
        run: |
          pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \
            jupyter nbconvert google-auth google-auth-oauthlib \
            google-api-python-client

      - name: Execute notebook
        env:
          USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
          USAJOBS_KEY: ${{ secrets.USAJOBS_KEY }}
          JOOBLE_KEY: ${{ secrets.JOOBLE_KEY }}
        run: |
          mkdir -p job_outputs job_work
          # Using the filename provided by user
          papermill "CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb" \
            ./executed_notebook.ipynb \
            --log-output \
            --no-progress-bar \
            -p DRIVE_BASE job_outputs

      - name: Upload outputs to Google Drive
        env:
          GDRIVE_FOLDER_ID: ${{ secrets.GDRIVE_FOLDER_ID }}
        run: python upload_to_drive.py

      - name: Archive executed notebook
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: executed-notebook-${{ github.run_id }}
          path: ./executed_notebook.ipynb
```

```javascript
// ── Google Apps Script: Weekly Job Collection ────────────────────────────────
const NOTEBOOK_FILE_ID = 'YOUR_NOTEBOOK_FILE_ID'; // ← paste here
const NOTIFY_EMAIL = Session.getActiveUser().getEmail();

function executeNotebook() {
  const token = ScriptApp.getOAuthToken();
  const url = `https://colab.research.google.com/v2/drives/drive/files/${NOTEBOOK_FILE_ID}:execute`;
  const opts = {
    method: 'post',
    headers: { Authorization: `Bearer ${token}`, 'Content-Type': 'application/json' },
    payload: JSON.stringify({ accelerator_type: 'NONE' }),
    muteHttpExceptions: true,
  };
  const resp = UrlFetchApp.fetch(url, opts);
  const code = resp.getResponseCode();
  Logger.log(`HTTP ${code} — ${resp.getContentText().substring(0, 200)}`);
  return code;
}

function weeklyJobCollection() {
  Logger.log('Run started: ' + new Date());
  const code = executeNotebook();

  const subject = code >= 200 && code < 300
    ? 'Job Pipeline Triggered'
    : `Job Pipeline FAILED (HTTP ${code})`;
  const body = code >= 200 && code < 300
    ? `Pipeline triggered at ${new Date()}. Check Drive/Research/JobPostings/outputs/ in ~35 min.`
    : `Trigger failed (HTTP ${code}). Verify notebook file ID and that notebook is saved in Drive.`;
  GmailApp.sendEmail(NOTIFY_EMAIL, subject, body);
}

// Run this once to create the Monday 6 AM trigger programmatically
function createWeeklyTrigger() {
  ScriptApp.getProjectTriggers()
    .filter(t => t.getHandlerFunction() === 'weeklyJobCollection')
    .forEach(t => ScriptApp.deleteTrigger(t));
  ScriptApp.newTrigger('weeklyJobCollection')
    .timeBased().onWeekDay(ScriptApp.WeekDay.MONDAY).atHour(6).create();
  Logger.log('Trigger created: every Monday at 6 AM.');
}
```

--## Section 9 — Incremental Update (Archive → Append)
**First run:** Sections 4–7 build the initial master.
**All subsequent runs:** uncomment `run_incremental_update()` at the bottom.
Steps per run:
1. Move prior `MASTER_*` files to `archive/`
2. Load prior master hash set as the dedup baseline
3. Filter fresh records (already in memory from Sections 4–5) to new-only
4. Back-fill O\\*NET skills for any pre-v3.1 records
5. Concat, dedup, save updated master

In [ ]:
def find_latest_master():
candidates = sorted(
[f for f in os.listdir(DRIVE_OUTPUTS) if f.startswith('MASTER_') and
f.endswith('.json')],
reverse=True)
return os.path.join(DRIVE_OUTPUTS, candidates[0]) if candidates else None

def archive_outputs():
moved = []
for fname in os.listdir(DRIVE_OUTPUTS):
if fname.startswith('MASTER_'):
src = os.path.join(DRIVE_OUTPUTS, fname)
dst = os.path.join(DRIVE_ARCHIVE, fname)
shutil.move(src, dst)
moved.append(fname)
print(f' Archived: {fname}')
return moved

def run_incremental_update():
global df_all, master
prior_path = find_latest_master()
if not prior_path:
print('No prior master — run Sections 4–7 first.')
return
print(f'Loading prior master: {os.path.basename(prior_path)}')
with open(prior_path, encoding='utf-8') as f:
prior_data = json.load(f)
prior_postings = prior_data['postings']
prior_hids
= {p['hash_id'] for p in prior_postings}
print(f' Prior records : {len(prior_postings)}')


print('Archiving prior files...')
archive_outputs()
new_raw = []
if 'usajobs_records' in dir(): new_raw.extend(usajobs_records)
if 'jooble_records' in dir(): new_raw.extend(jooble_records)
if not new_raw:
print('No fresh records in memory — run Sections 4 and 5 first.')
return
new_only = [r for r in new_raw if r['hash_id'] not in prior_hids]
print(f' Fresh fetched : {len(new_raw)}')
print(f' New only
: {len(new_only)}')
# Back-fill O*NET for pre-v3.1 prior records
backfilled = 0
for p in prior_postings:
if not p.get('onet_skills'):
p['onet_skills']
= map_onet_skills(p.get('title',''),
p.get('description',''), p.get('required_skills',[]))
p['technical_skills'] = p.get('required_skills', [])
backfilled += 1
if backfilled:
print(f' Back-filled O*NET for {backfilled} prior records')
if not new_only:
print('No new records — master is current.')
return
FIELDS = ['hash_id','job_title','company','location','required_skills',
'technical_skills','onet_skills','domain','era','publication_date',
'application_close','salary','job_type','grade','job_url',
'description','relevance_score','matched_terms','relevance_tier',
'source','search_keyword']
prior_rows = []
for p in prior_postings:
row = {f: p.get(f if f != 'publication_date' else 'posted_date', p.get(f, ''))
for f in FIELDS}
row['job_title'] = p.get('title', p.get('job_title', ''))
prior_rows.append(row)
df_all = pd.concat([pd.DataFrame(prior_rows), pd.DataFrame(new_only)],
ignore_index=True)
df_all = (df_all.sort_values('relevance_score', ascending=False)
.drop_duplicates(subset='hash_id', keep='first')
.reset_index(drop=True))
df_all.insert(0, 'id', range(1, len(df_all) + 1))
net_new = len(df_all) - len(prior_postings)
print(f' Updated total : {len(df_all)} records (+{net_new} net new)')


master
= build_master_json(df_all)
loc_json = os.path.join(LOCAL_WORK, MASTER_JSON)
with open(loc_json, 'w', encoding='utf-8') as f:
json.dump(master, f, indent=2, ensure_ascii=False)
save_to_drive(loc_json, DRIVE_OUTPUTS, MASTER_JSON)
write_run_log(df_all)
print(f'Incremental update complete. Re-run Section 7 to regenerate CSV + XLSX.')

print("Incremental update ready.")
print(" First run : Sections 4–7 build the initial master.")
print(" Subsequent : uncomment run_incremental_update() below.")
# ── Uncomment for all runs AFTER the first master is built ────────────────────
# run_incremental_update()

In [1]:
print('gcloud iam service-accounts create github-pipeline --display-name="GitHub Actions Job Pipeline" --project=project-bf62429c-050a-4ead-b92')

gcloud iam service-accounts create github-pipeline --display-name="GitHub Actions Job Pipeline" --project=project-bf62429c-050a-4ead-b92
